In [9]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate

In [10]:
print("⏳ Đang tải dataset...")
dataset = load_dataset("imdb")

⏳ Đang tải dataset...


In [11]:
train_dataset = dataset["train"].shuffle(seed=42).select(range(2000))
eval_dataset = dataset["test"].shuffle(seed=42).select(range(500))

In [12]:
model_checkpoint = "bert-base-uncased" # Dùng Bert gốc cho ổn định nhất
print(f"🚀 Đang tải model: {model_checkpoint}...")

🚀 Đang tải model: bert-base-uncased...


In [13]:
try:
    tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
except:
    tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, use_fast=False)

model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [14]:
def tokenize_function(examples):
    # max_length=256 giúp model đọc được câu dài hơn -> Accuracy cao hơn 128
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=256)

print("⏳ Đang xử lý dữ liệu (Tokenization)...")

# SỬA LỖI: Dùng đúng tên biến train_dataset đã khai báo ở trên
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_eval = eval_dataset.map(tokenize_function, batched=True)

⏳ Đang xử lý dữ liệu (Tokenization)...


Map: 100%|██████████| 500/500 [00:00<00:00, 3549.43 examples/s]


In [15]:
training_args = TrainingArguments(
    output_dir="bert_high_acc",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"
)

In [16]:
metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [17]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    compute_metrics=compute_metrics,
)

print("🚀 Bắt đầu Training...")
trainer.train()

🚀 Bắt đầu Training...


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.301545,0.880000
2,0.322000,0.550879,0.862000
3,0.322000,0.422013,0.898000


TrainOutput(global_step=750, training_loss=0.2575197550455729, metrics={'train_runtime': 580.4525, 'train_samples_per_second': 10.337, 'train_steps_per_second': 1.292, 'total_flos': 789333166080000.0, 'train_loss': 0.2575197550455729, 'epoch': 3.0})

In [18]:
print("\n📊 Kết quả đánh giá trên tập test:")
metrics = trainer.evaluate()
print(f"\n🏆 Accuracy cuối cùng: {metrics['eval_accuracy']*100:.2f}%")


📊 Kết quả đánh giá trên tập test:



🏆 Accuracy cuối cùng: 88.00%


In [19]:
print("\n🧪 Test thử model:")
text = "This movie is absolutely amazing and fantastic!" # Câu này nên là Positive
inputs = tokenizer(text, return_tensors="pt").to(model.device)
with torch.no_grad():
    logits = model(**inputs).logits
predicted_class_id = logits.argmax().item()
print(f"Text: '{text}'")
print(f"Label: {'POSITIVE' if predicted_class_id == 1 else 'NEGATIVE'}")


🧪 Test thử model:
Text: 'This movie is absolutely amazing and fantastic!'
Label: POSITIVE
